# Локальный стекинг отгрузок
Запустите из корня проекта в окружении с pandas, pyarrow, scipy, scikit-learn, prophet, catboost и XGBoost с поддержкой CUDA. Модели обучаются по очереди. Полный запуск использует восемь горизонтов. Для пробного запуска задайте HORIZONS = (2,). В таком режиме получится таблица ошибок для h2.

Обучающие примеры деревьев используют все получасовые старты. Строки для оценки и метамодели используют только прогнозы из 10:30. Два последовательных 14-дневных блока образуют train метамодели. На каждом блоке базовые модели обучаются на более ранних метках. Следующие 14 дней - val для выбора регуляризации метамодели, последние 14 - локальный test. На val и test прогнозирует один набор базовых моделей, обученный до val.

Для Prophet используется одна модель на маршрут и границу обучения. Она выдаёт сразу все нужные даты и горизонты блока. Поэтому вдали от границы её прогноз опирается на более старую историю, чем сезонный прогноз. Это компромисс по времени локального запуска.


In [1]:
import faulthandler

trace_file = open("import_trace.log", "w", buffering=1)
faulthandler.dump_traceback_later(
    30, repeat=True, file=trace_file
)

In [2]:
faulthandler.cancel_dump_traceback_later()
trace_file.close()

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data")
TRAIN_FILE = "train_solo_track.parquet"
TEST_FILE = "test_solo_track.parquet"
if not (DATA / TRAIN_FILE).is_file() and Path("/kaggle/input").exists():
    matches = [p.parent for p in Path("/kaggle/input").rglob(TRAIN_FILE)
               if (p.parent / TEST_FILE).is_file()]
    if len(matches) != 1:
        raise ValueError(f"Укажите DATA вручную. Найденные папки: {matches}")
    DATA = matches[0]
OUT = Path("artifacts/local_stack/features")
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "target_1h"
STATUSES = [f"status_{i}" for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
HORIZONS = tuple(range(1, 9))
SCALE = 100_000
if not HORIZONS or len(set(HORIZONS)) != len(HORIZONS) or not set(HORIZONS) <= set(range(1, 9)):
    raise ValueError("HORIZONS должен содержать разные горизонты от 1 до 8")
HORIZONS = tuple(sorted(HORIZONS))
from prophet import Prophet
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
import gc
if get_gpu_device_count() < 1:
    raise RuntimeError("CatBoost не видит GPU. Проверьте драйвер NVIDIA.")

## Получасовая сетка
Проверяем полноту панели перед использованием позиционных лагов. Строки массивов - моменты времени, столбцы - маршруты.


In [4]:
history = pd.read_parquet(DATA / TRAIN_FILE).sort_values(["timestamp", "route_id"])
test = pd.read_parquet(DATA / TEST_FILE)
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError("В данных есть пропуски")
if not (not history.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторяется ключ маршрута и времени")
if not (times.equals(pd.date_range(times[0], times[-1], freq=STEP))):
    raise ValueError("Нарушена получасовая сетка")
if not (len(history) == len(times) * len(routes)):
    raise ValueError("Неполная панель маршрутов")
if not (test.id.is_unique and not test.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторы идентификаторов или ключей теста")
end = times[-1]
if not ((end.hour, end.minute) == (10, 30)):
    raise ValueError("Последняя точка истории должна быть в 10:30")
expected = pd.MultiIndex.from_product([routes, [end + h*STEP for h in range(1, 9)]],
                                      names=["route_id", "timestamp"])
if not (expected.difference(pd.MultiIndex.from_frame(test[["route_id", "timestamp"]])).empty):
    raise ValueError("Тест не покрывает все ожидаемые маршруты и горизонты")
if not (len(test) == len(expected)):
    raise ValueError("Число строк теста отличается от ожидаемого")
values = {name: history[name].to_numpy().reshape(len(times), len(routes))
          for name in [TARGET, *STATUSES]}
del history

## Исторические признаки
Для каждого получаса используем данные до этого момента включительно. В начале истории лаги без наблюдений остаются NaN, средние считаются по доступным точкам.


In [5]:
import pyarrow as pa
import pyarrow.parquet as pq

target = values[TARGET]
def past(array, positions):
    result = array[np.maximum(positions, 0)].astype(np.float32)
    result[positions < 0] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0])
    count = np.zeros_like(total)
    for matrix in matrices:
        total += np.nan_to_num(matrix)
        count += np.isfinite(matrix)
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f"target_back_{k}"] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f"{name}_mean_{steps}"] = means[name, steps][origins]
        if name in STATUSES:
            common[f"{name}_latest"] = array[origins]
            common[f"{name}_change_1h"] = array[origins] - past(array, origins - 2)
    common["target_vs_mean_6h"] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day*48) for day in range(1, 8)])
    common["morning_level_ratio"] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common

## Признаки слота и сохранение
Каждая строка соответствует одному маршруту, дате и горизонту. Метка y доступна только для истории. Колонки id, cutoff, timestamp и y служебные.


In [6]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({
        "cutoff": times[origins].repeat(len(routes)),
        "timestamp": dates.repeat(len(routes)),
        "route_id": np.tile(routes, len(origins)),
    })
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    for name, index, period in [
        ("weekday", dates.dayofweek, 7),
        ("half_hour", dates.hour * 2 + dates.minute // 30 + 1, 48),
    ]:
        phase = index if name == "weekday" else index - 1
        angle = 2 * np.pi * phase / period
        frame[name + "_sin"] = np.repeat(np.sin(angle), len(routes))
        frame[name + "_cos"] = np.repeat(np.cos(angle), len(routes))
    for days in (1, 2, 7):
        frame[f"slot_days_back_{days}"] = past(target, slot - days*48).ravel()
    for name, days in [("slot_daily7", range(1, 8)), ("slot_weekly4", (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d*48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day*48)
        adjusted = past(target, slot - day*48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame["profile28"] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = np.maximum(0, total - upper), np.minimum(total, total - lower)
        seasonal = frame[["slot_daily7", "slot_weekly4"]].mean(axis=1).to_numpy()
        frame["overlap"] = np.where((lower <= upper).ravel(),
                                    (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame["overlap"] = np.nan
    numeric = frame.columns.difference(["cutoff", "timestamp", "route_id"])
    return frame.astype({name: "float32" for name in numeric})

def write_horizon(h):
    stop = len(times) - h
    path = OUT / f"history_h{h}.parquet"
    temporary = path.with_suffix(".partial")
    writer = None
    try:
        for start in range(0, stop, 48):
            origins = np.arange(start, min(start + 48, stop))
            frame = horizon_dataset(h, origins)
            frame["y"] = target[origins + h].ravel()
            table = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h*STEP)]
    future = requests.merge(features, on=["route_id", "timestamp"], how="left", validate="one_to_one")
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError("Ошибка сопоставления тестовых строк")
    future.to_parquet(OUT / f"submission_h{h}.parquet", index=False)
    print(f"h={h}: история {stop * len(routes):,}, тест {len(future):,}")

for h in HORIZONS:
    write_horizon(h)

del values, target, means, level24

C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267

h=1: история 4,629,000, тест 1,000


C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267

h=2: история 4,628,000, тест 1,000


C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267

h=3: история 4,627,000, тест 1,000


... (6 outputs left out)

C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267

h=7: история 4,623,000, тест 1,000


C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267987.py:12: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  ("weekday", dates.dayofweek, 7),
C:\Users\artem\AppData\Local\Temp\ipykernel_15484\371267

h=8: история 4,622,000, тест 1,000


## Временные блоки и метрики
Train в итоговой таблице означает строки, на которых обучена метамодель. Для базовых моделей это прогнозы на будущие относительно их обучения даты. Метрика конкурса считается сразу по всем строкам соответствующей части. Вторая оценка усредняет отдельные дневные метрики.


In [7]:
dates = times[(times.hour == 10) & (times.minute == 30) & (times + STEP * 8 <= end)]
if len(dates) < 85:
    raise ValueError("Для выбранных четырёх блоков и сезонных признаков недостаточно истории")
blocks = {"oof1": dates[-56:-42], "oof2": dates[-42:-28],
          "evaluation": dates[-28:], "final": pd.DatetimeIndex([end])}
methods = ["seasonal", "prophet", "ridge", "forest", "xgboost", "catboost"]
def metric(frame, pred):
    delta = np.asarray(pred) - frame.y.to_numpy()
    daily = pd.DataFrame({"cutoff": frame.cutoff, "volume": frame.y, "abs": np.abs(delta),
                          "error": delta}).groupby("cutoff").sum()
    total = daily.volume.sum()
    wape = daily["abs"].sum() / total
    bias = abs(daily.error.sum()) / total
    return {"строки": len(frame), "WAPE": wape, "|Relative Bias|": bias,
            "метрика конкурса": wape + bias,
            "по дням": ((daily["abs"] + daily.error.abs()) / daily.volume).mean()}

def report(frame, method):
    return pd.DataFrame([dict(метод=method, выборка=split, **metric(rows, rows[method]))
                         for split, rows in frame.groupby("split", sort=False)])

## Прогноз Prophet по маршрутам
Граница истории для промежуточных моделей - начало первого дня окна. При финальном обучении доступны факты до 10:30 включительно. Маленькие таблицы прогнозов сохраняются на диск.


In [8]:
def prophet_block(block):
    days = blocks[block]
    raw = pd.read_parquet(DATA / TRAIN_FILE, columns=["route_id", "timestamp", TARGET])
    pieces = []
    limit = end if block == "final" else days[0].normalize() - STEP
    requested = pd.DataFrame([(d, h, d + h*STEP) for d in days for h in HORIZONS],
                             columns=["cutoff", "horizon", "ds"])
    for route, series in raw.groupby("route_id", sort=False):
        fit = series.loc[series.timestamp <= limit, ["timestamp", TARGET]].copy()
        fit.columns = ["ds", "y"]
        model = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False,
                        uncertainty_samples=0)
        model.fit(fit)
        result = requested.copy()
        result["route_id"] = route
        result["prophet"] = model.predict(requested[["ds"]]).yhat.to_numpy().clip(min=0)
        pieces.append(result.drop(columns="ds"))
        if len(pieces) % 100 == 0:
            print(block, "обработано маршрутов:", len(pieces))
        del model
    pd.concat(pieces, ignore_index=True).to_parquet(OUT.parent / f"prophet_{block}.parquet", index=False)
    print(block, len(pieces), "маршрутов")

for block in ("oof1", "oof2", "evaluation"):
    prophet_block(block)

c:\Users\artem\projects\experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
03:39:23 - cmdstanpy - INFO - Chain [1] start processing
03:39:23 - cmdstanpy - INFO - Chain [1] done processing
03:39:23 - cmdstanpy - INFO - Chain [1] start processing
03:39:23 - cmdstanpy - INFO - Chain [1] done processing
03:39:24 - cmdstanpy - INFO - Chain [1] start processing
03:39:24 - cmdstanpy - INFO - Chain [1] done processing
03:39:24 - cmdstanpy - INFO - Chain [1] start processing
03:39:24 - cmdstanpy - INFO - Chain [1] done processing
03:39:24 - cmdstanpy - INFO - Chain [1] start processing
03:39:24 - cmdstanpy - INFO - Chain [1] done processing
03:39:24 - cmdstanpy - INFO - Chain [1] start processing
03:39:24 - cmdstanpy - INFO - Chain [1] done processing
03:39:24 - cmdstanpy - INFO - Chain [1] start processing


oof1 обработано маршрутов: 100


03:39:47 - cmdstanpy - INFO - Chain [1] start processing
03:39:47 - cmdstanpy - INFO - Chain [1] done processing
03:39:47 - cmdstanpy - INFO - Chain [1] start processing
03:39:47 - cmdstanpy - INFO - Chain [1] done processing
03:39:47 - cmdstanpy - INFO - Chain [1] start processing
03:39:47 - cmdstanpy - INFO - Chain [1] done processing
03:39:48 - cmdstanpy - INFO - Chain [1] start processing
03:39:48 - cmdstanpy - INFO - Chain [1] done processing
03:39:48 - cmdstanpy - INFO - Chain [1] start processing
03:39:48 - cmdstanpy - INFO - Chain [1] done processing
03:39:48 - cmdstanpy - INFO - Chain [1] start processing
03:39:48 - cmdstanpy - INFO - Chain [1] done processing
03:39:48 - cmdstanpy - INFO - Chain [1] start processing
03:39:48 - cmdstanpy - INFO - Chain [1] done processing
03:39:49 - cmdstanpy - INFO - Chain [1] start processing
03:39:49 - cmdstanpy - INFO - Chain [1] done processing
03:39:49 - cmdstanpy - INFO - Chain [1] start processing
03:39:49 - cmdstanpy - INFO - Chain [1]

oof1 обработано маршрутов: 200


03:40:24 - cmdstanpy - INFO - Chain [1] start processing
03:40:24 - cmdstanpy - INFO - Chain [1] done processing
03:40:24 - cmdstanpy - INFO - Chain [1] start processing
03:40:24 - cmdstanpy - INFO - Chain [1] done processing
03:40:25 - cmdstanpy - INFO - Chain [1] start processing
03:40:25 - cmdstanpy - INFO - Chain [1] done processing
03:40:25 - cmdstanpy - INFO - Chain [1] start processing
03:40:25 - cmdstanpy - INFO - Chain [1] done processing
03:40:26 - cmdstanpy - INFO - Chain [1] start processing
03:40:26 - cmdstanpy - INFO - Chain [1] done processing
03:40:26 - cmdstanpy - INFO - Chain [1] start processing
03:40:26 - cmdstanpy - INFO - Chain [1] done processing
03:40:26 - cmdstanpy - INFO - Chain [1] start processing
03:40:26 - cmdstanpy - INFO - Chain [1] done processing
03:40:27 - cmdstanpy - INFO - Chain [1] start processing
03:40:27 - cmdstanpy - INFO - Chain [1] done processing
03:40:27 - cmdstanpy - INFO - Chain [1] start processing
03:40:27 - cmdstanpy - INFO - Chain [1]

oof1 обработано маршрутов: 300


... (50 outputs left out)

03:51:02 - cmdstanpy - INFO - Chain [1] done processing
03:51:02 - cmdstanpy - INFO - Chain [1] start processing
03:51:02 - cmdstanpy - INFO - Chain [1] done processing
03:51:02 - cmdstanpy - INFO - Chain [1] start processing
03:51:02 - cmdstanpy - INFO - Chain [1] done processing
03:51:02 - cmdstanpy - INFO - Chain [1] start processing
03:51:03 - cmdstanpy - INFO - Chain [1] done processing
03:51:03 - cmdstanpy - INFO - Chain [1] start processing
03:51:03 - cmdstanpy - INFO - Chain [1] done processing
03:51:03 - cmdstanpy - INFO - Chain [1] start processing
03:51:03 - cmdstanpy - INFO - Chain [1] done processing
03:51:03 - cmdstanpy - INFO - Chain [1] start processing
03:51:03 - cmdstanpy - INFO - Chain [1] done processing
03:51:04 - cmdstanpy - INFO - Chain [1] start processing
03:51:04 - cmdstanpy - INFO - Chain [1] done processing
03:51:04 - cmdstanpy - INFO - Chain [1] start processing
03:51:04 - cmdstanpy - INFO - Chain [1] done processing
03:51:04 - cmdstanpy - INFO - Chain [1] 

evaluation обработано маршрутов: 900


03:51:30 - cmdstanpy - INFO - Chain [1] done processing
03:51:30 - cmdstanpy - INFO - Chain [1] start processing
03:51:30 - cmdstanpy - INFO - Chain [1] done processing
03:51:31 - cmdstanpy - INFO - Chain [1] start processing
03:51:31 - cmdstanpy - INFO - Chain [1] done processing
03:51:31 - cmdstanpy - INFO - Chain [1] start processing
03:51:31 - cmdstanpy - INFO - Chain [1] done processing
03:51:31 - cmdstanpy - INFO - Chain [1] start processing
03:51:31 - cmdstanpy - INFO - Chain [1] done processing
03:51:31 - cmdstanpy - INFO - Chain [1] start processing
03:51:32 - cmdstanpy - INFO - Chain [1] done processing
03:51:32 - cmdstanpy - INFO - Chain [1] start processing
03:51:32 - cmdstanpy - INFO - Chain [1] done processing
03:51:32 - cmdstanpy - INFO - Chain [1] start processing
03:51:32 - cmdstanpy - INFO - Chain [1] done processing
03:51:32 - cmdstanpy - INFO - Chain [1] start processing
03:51:32 - cmdstanpy - INFO - Chain [1] done processing
03:51:33 - cmdstanpy - INFO - Chain [1] 

evaluation обработано маршрутов: 1000
evaluation 1000 маршрутов


## Сезонный прогноз, Ridge организаторов и деревья
Сезонный прогноз - полусумма средних того же слота за семь дней и четыре недели. Ridge использует шесть последних статусов, разреженный one-hot маршрута и окно последних 14 дней доступной обучающей истории. Лес использует ограниченную подвыборку для каждого дерева. Оба бустинга обучаются на GPU на масштабированном таргете.


In [9]:
def fit_predict(method, train, query, features):
    if method == "ridge":
        statuses = [f"status_{i}_latest" for i in range(1, 7)]
        columns = ["route_id", *statuses]
        encoder = ColumnTransformer([("route", OneHotEncoder(handle_unknown="ignore"), ["route_id"]),
                                     ("statuses", StandardScaler(), statuses)])
        model = make_pipeline(encoder, Ridge(alpha=4.0, solver="lsqr"))
        train = train.loc[train.cutoff >= train.cutoff.max() - pd.Timedelta(days=14)]
    elif method == "forest":
        columns = [c for c in features if c != "route_id"]
        model = RandomForestRegressor(n_estimators=80, max_depth=12, min_samples_leaf=20,
                                      max_samples=0.2, n_jobs=2, random_state=42)
    elif method == "xgboost":
        columns = [c for c in features if c != "route_id"]
        model = XGBRegressor(device="cuda", tree_method="hist", n_estimators=600, max_depth=6,
                             learning_rate=0.05, n_jobs=2, random_state=42)
    else:
        columns = features
        model = CatBoostRegressor(task_type="GPU", devices="0", loss_function="MAE", iterations=1500,
                                  depth=7, learning_rate=0.05, cat_features=["route_id"],
                                  random_seed=42, allow_writing_files=False, verbose=250)
    model.fit(train[columns], train.y / SCALE)
    return (model.predict(query[columns]) * SCALE).clip(min=0)

def predict_block(h, block):
    days = blocks[block]
    path = OUT / f"history_h{h}.parquet"
    if block == "final":
        query = pd.read_parquet(OUT / f"submission_h{h}.parquet")
        filters = None
    else:
        query = pd.read_parquet(path, filters=[("cutoff", "in", days.to_list())])
        filters = [("timestamp", "<", days[0].normalize())]
    keys = ["cutoff", "route_id", "timestamp", "id" if block == "final" else "y"]
    result = query[keys].copy()
    result["horizon"] = h
    result["split"] = ("submission" if block == "final" else "train" if block.startswith("oof")
                       else np.where(result.cutoff < dates[-14], "val", "test"))
    result["seasonal"] = query[["slot_daily7", "slot_weekly4"]].mean(axis=1)
    prophet = pd.read_parquet(OUT.parent / f"prophet_{block}.parquet", filters=[("horizon", "==", h)])
    result = result.merge(prophet, on=["cutoff", "route_id", "horizon"], how="left", validate="one_to_one")
    features = [c for c in query if c not in {"id", "cutoff", "timestamp", "y", "overlap", "profile28"}]
    train = pd.read_parquet(path, columns=["cutoff", *features, "y"], filters=filters)
    for method in methods[2:]:
        result[method] = fit_predict(method, train, query, features)
        if block == "evaluation":
            prior = pd.concat([part for part in evaluation_parts if part.horizon.iloc[0] == h])
            display(report(pd.concat([prior, result], ignore_index=True), method))
        elif block != "final":
            display(report(result, method))
        else:
            display(results.loc[results["метод"].eq(method)])
        gc.collect()
    if len(result) != len(days)*len(routes) or not np.isfinite(result[methods]).all().all():
        raise ValueError(f"Неполные прогнозы: {block}, h={h}")
    return result

evaluation_parts = []
for h in HORIZONS:
    for block in ("oof1", "oof2", "evaluation"):
        evaluation_parts.append(predict_block(h, block))
scores = pd.concat(evaluation_parts, ignore_index=True)
del evaluation_parts
scores.to_parquet(OUT.parent / "base_predictions.parquet", index=False)

,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
0,ridge,train,14000,0.323168,0.020857,0.344024,0.374414


,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
0,forest,train,14000,0.273788,0.035529,0.309317,0.326869


c:\Users\artem\projects\experiments\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [04:00:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
0,xgboost,train,14000,0.266194,0.032635,0.298829,0.31634


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1522758	total: 74ms	remaining: 1m 50s
250:	learn: 0.6458086	total: 10.4s	remaining: 51.7s
500:	learn: 0.6324841	total: 20.6s	remaining: 41s
750:	learn: 0.6261193	total: 30.6s	remaining: 30.5s
1000:	learn: 0.6218959	total: 40.6s	remaining: 20.2s
1250:	learn: 0.6189721	total: 50.6s	remaining: 10.1s
1499:	learn: 0.6167127	total: 1m	remaining: 0us


... (135 outputs left out)

,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
0,xgboost,train,28000,0.331511,0.022882,0.354392,0.364858
1,xgboost,val,14000,0.340099,0.004832,0.344931,0.372868
2,xgboost,test,14000,0.336360,0.028764,0.365124,0.380882


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2436621	total: 124ms	remaining: 3m 5s
250:	learn: 0.8962240	total: 19.5s	remaining: 1m 37s
500:	learn: 0.8926238	total: 38.6s	remaining: 1m 16s
750:	learn: 0.8911555	total: 58.1s	remaining: 57.9s
1000:	learn: 0.8901070	total: 1m 17s	remaining: 38.7s
1250:	learn: 0.8892463	total: 1m 37s	remaining: 19.4s
1499:	learn: 0.8884992	total: 1m 56s	remaining: 0us


,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
0,catboost,train,28000,0.331513,0.067018,0.398531,0.397528
1,catboost,val,14000,0.340586,0.051407,0.391993,0.391598
2,catboost,test,14000,0.338353,0.071616,0.409969,0.408842


## Стекинг
Для каждого горизонта Ridge объединяет шесть прогнозов в исходных единицах после общего масштабирования. Метамодель обучается на двух OOF-блоках train. Общий alpha выбираем по средней дневной конкурсной метрике на val сразу по всем горизонтам. После фиксации настроек оцениваем test.


In [10]:
def stack(train, query, alpha):
    prediction = pd.Series(index=query.index, dtype=float)
    for h, part in query.groupby("horizon"):
        fit = train.loc[train.horizon.eq(h)]
        model = Ridge(alpha=alpha)
        model.fit(fit[methods] / SCALE, fit.y / SCALE)
        prediction.loc[part.index] = (model.predict(part[methods] / SCALE) * SCALE).clip(min=0)
    return prediction

meta_train = scores.loc[scores.split.eq("train")]
val = scores.loc[scores.split.eq("val")]
alphas = (1, 100, 1000, 10000)
alpha = min(alphas, key=lambda a: metric(val, stack(meta_train, val, a))["по дням"])
scores["stacking"] = stack(meta_train, scores, alpha)
print("Выбранный alpha:", alpha)
results = pd.concat([report(scores, name) for name in [*methods, "stacking"]], ignore_index=True)
display(results)

Выбранный alpha: 1


,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
0,seasonal,train,224000,0.350512,5.071388e-02,0.401226,0.401149
1,seasonal,val,112000,0.363094,1.658622e-02,0.379681,0.385138
2,seasonal,test,112000,0.354692,1.455218e-02,0.369244,0.381694
3,prophet,train,224000,0.344445,2.556883e-03,0.347001,0.372768
4,prophet,val,112000,0.362730,3.171146e-02,0.394441,0.403734
5,prophet,test,112000,0.391909,7.017960e-02,0.462089,0.463228
6,ridge,train,224000,0.334568,3.783110e-02,0.372400,0.380451
7,ridge,val,112000,0.341304,1.738861e-02,0.358692,0.364982
8,ridge,test,112000,0.343774,2.247805e-02,0.366252,0.366677
9,forest,train,224000,0.322741,1.168702e-02,0.334428,0.341728


## Финальное переобучение и сабмит
Базовые модели переобучаются с данными до последней доступной точки. Для Ridge сохраняем правило последних 14 дней. Метамодель переобучается на всех прогнозах исторических проверок. Независимая оценка test в последней таблице получена до этого этапа.


In [11]:
prophet_block("final")
future = pd.concat([predict_block(h, "final") for h in HORIZONS], ignore_index=True)
future["y_pred"] = stack(scores, future, alpha)
destination = OUT.parent / "stacking_forecast.parquet"
future.to_parquet(destination, index=False)
if len(HORIZONS) == 8:
    submission = test[["id"]].merge(future[["id", "y_pred"]], on="id", how="left", validate="one_to_one")
    if not np.isfinite(submission.y_pred).all():
        raise ValueError("Пропущены прогнозы сабмита")
    submission.to_csv(OUT.parent / "submission.csv", index=False)
print("Результаты:", OUT.parent.resolve())

09:04:59 - cmdstanpy - INFO - Chain [1] start processing
09:04:59 - cmdstanpy - INFO - Chain [1] done processing
09:04:59 - cmdstanpy - INFO - Chain [1] start processing
09:04:59 - cmdstanpy - INFO - Chain [1] done processing
09:05:00 - cmdstanpy - INFO - Chain [1] start processing
09:05:00 - cmdstanpy - INFO - Chain [1] done processing
09:05:00 - cmdstanpy - INFO - Chain [1] start processing
09:05:00 - cmdstanpy - INFO - Chain [1] done processing
09:05:01 - cmdstanpy - INFO - Chain [1] start processing
09:05:01 - cmdstanpy - INFO - Chain [1] done processing
09:05:01 - cmdstanpy - INFO - Chain [1] start processing
09:05:01 - cmdstanpy - INFO - Chain [1] done processing
09:05:01 - cmdstanpy - INFO - Chain [1] start processing
09:05:01 - cmdstanpy - INFO - Chain [1] done processing
09:05:02 - cmdstanpy - INFO - Chain [1] start processing
09:05:02 - cmdstanpy - INFO - Chain [1] done processing
09:05:02 - cmdstanpy - INFO - Chain [1] start processing
09:05:02 - cmdstanpy - INFO - Chain [1]

final обработано маршрутов: 100


09:05:37 - cmdstanpy - INFO - Chain [1] done processing
09:05:37 - cmdstanpy - INFO - Chain [1] start processing
09:05:37 - cmdstanpy - INFO - Chain [1] done processing
09:05:37 - cmdstanpy - INFO - Chain [1] start processing
09:05:37 - cmdstanpy - INFO - Chain [1] done processing
09:05:37 - cmdstanpy - INFO - Chain [1] start processing
09:05:38 - cmdstanpy - INFO - Chain [1] done processing
09:05:38 - cmdstanpy - INFO - Chain [1] start processing
09:05:38 - cmdstanpy - INFO - Chain [1] done processing
09:05:38 - cmdstanpy - INFO - Chain [1] start processing
09:05:39 - cmdstanpy - INFO - Chain [1] done processing
09:05:39 - cmdstanpy - INFO - Chain [1] start processing
09:05:39 - cmdstanpy - INFO - Chain [1] done processing
09:05:39 - cmdstanpy - INFO - Chain [1] start processing
09:05:39 - cmdstanpy - INFO - Chain [1] done processing
09:05:39 - cmdstanpy - INFO - Chain [1] start processing
09:05:40 - cmdstanpy - INFO - Chain [1] done processing
09:05:40 - cmdstanpy - INFO - Chain [1] 

final обработано маршрутов: 200


09:06:14 - cmdstanpy - INFO - Chain [1] done processing
09:06:14 - cmdstanpy - INFO - Chain [1] start processing
09:06:14 - cmdstanpy - INFO - Chain [1] done processing
09:06:14 - cmdstanpy - INFO - Chain [1] start processing
09:06:14 - cmdstanpy - INFO - Chain [1] done processing
09:06:15 - cmdstanpy - INFO - Chain [1] start processing
09:06:15 - cmdstanpy - INFO - Chain [1] done processing
09:06:15 - cmdstanpy - INFO - Chain [1] start processing
09:06:15 - cmdstanpy - INFO - Chain [1] done processing
09:06:15 - cmdstanpy - INFO - Chain [1] start processing
09:06:16 - cmdstanpy - INFO - Chain [1] done processing
09:06:16 - cmdstanpy - INFO - Chain [1] start processing
09:06:16 - cmdstanpy - INFO - Chain [1] done processing
09:06:16 - cmdstanpy - INFO - Chain [1] start processing
09:06:16 - cmdstanpy - INFO - Chain [1] done processing
09:06:17 - cmdstanpy - INFO - Chain [1] start processing
09:06:17 - cmdstanpy - INFO - Chain [1] done processing
09:06:17 - cmdstanpy - INFO - Chain [1] 

final обработано маршрутов: 300


... (59 outputs left out)

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2723008	total: 264ms	remaining: 6m 36s
250:	learn: 0.9167624	total: 30.6s	remaining: 2m 32s
500:	learn: 0.9131194	total: 1m	remaining: 2m
750:	learn: 0.9115969	total: 1m 30s	remaining: 1m 30s
1000:	learn: 0.9105435	total: 2m 1s	remaining: 1m
1250:	learn: 0.9097171	total: 2m 31s	remaining: 30.2s
1499:	learn: 0.9090353	total: 3m 2s	remaining: 0us


,метод,выборка,строки,WAPE,|Relative Bias|,метрика конкурса,по дням
15,catboost,train,224000,0.319626,0.048213,0.367839,0.368026
16,catboost,val,112000,0.324911,0.040252,0.365162,0.365069
17,catboost,test,112000,0.322822,0.060047,0.382869,0.382612


Результаты: C:\Users\artem\projects\experiments\artifacts\local_stack


## Итоговая таблица ошибок
Каждому методу соответствуют оценки train, val и test на одинаковых маршрутах, датах и горизонтах. Для стекинга train - оценка на обучающих строках метамодели. Независимый test оценён до финального переобучения.


In [12]:
results.to_csv(OUT.parent / "metrics.csv", index=False)
summary = results.pivot(index="метод", columns="выборка", values="метрика конкурса")
display(summary.loc[[*methods, "stacking"], ["train", "val", "test"]])

выборка,train,val,test
метод,,,
seasonal,0.401226,0.379681,0.369244
prophet,0.347001,0.394441,0.462089
ridge,0.372400,0.358692,0.366252
forest,0.334428,0.330295,0.344543
xgboost,0.324531,0.332502,0.337276
catboost,0.367839,0.365162,0.382869
stacking,0.320196,0.339848,0.328478
